# 📘 Notebook 14: Building the Warehouse

### Course: *From One Table to a Cluster: Working with Data at Scale*
**Instructor:** Ioannis Tsioulis

*Module E: A Real Warehouse, End to End · Notebook 2 of 6 in this module · (14 of 18 overall)*

---

> 🧭 **How to use this notebook.** Run the cells in order, from top to bottom: click a cell and press **Shift+Enter**. After most cells there is a short paragraph headed **What you should see**: compare it with your own output before you move on. If your numbers differ, stop and find out why, because that is how data work is learned. The exercises at the end start easy and get harder: first **warm-up** questions, then **think it through** questions (what would you run, and why?), then **technical** tasks. Try each one before you open its solution.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

- Use **SQLite from Python**: connect, create a table, insert rows, ask questions, and read the answers into pandas
- Turn profiled CSV files into a warehouse with **primary keys**, **foreign keys** and a stated grain
- Load the tables in the order that the keys require, and prove that every key holds
- Recognise a **snowflaked** dimension and a **role-playing** dimension, and say what each costs
- Hide the joins behind a **view** and answer business questions by rolling up, drilling down and slicing
- Measure what an **index** and a **summary table** buy, and say when each one helps

> **Prerequisites:** Notebook 13 (the profile of this warehouse). It helps to have seen Notebooks 01 to 04 (SQL, joins, indexes, the star schema), but this notebook starts SQLite from the beginning.
>
> 🔭 **Why this notebook matters:** in Notebook 13 we *looked at* the data with pandas. Now we put it into a real database, the job a data engineer does in the first week of a project: create the tables, load them, make the database **enforce** the rules we checked by hand, and make it easy and fast for analysts to ask questions. We use **SQLite**, a complete database that lives in a single file and comes built into Python. Cloud warehouses such as Amazon Redshift, Azure Synapse or Google BigQuery accept almost the same `CREATE TABLE` statements and the same queries, so what you build here is a small but faithful copy of the real thing.

> 📦 **Libraries used in this notebook.** None that need installing: **sqlite3** (part of Python) and **pandas**.

> 🗃️ **The data.** This module works with **AdventureWorks**, the sample data warehouse that Microsoft publishes for learning SQL Server: the online sales of an imaginary bicycle company, with its customers, products and dates. It is open data (MIT licence) and much closer to a real warehouse than anything we could invent. The cell below downloads it from Microsoft's GitHub repository (about 17 MB, once per session) and saves each table as an ordinary CSV file.

In [ ]:
import io
import os
import re
import urllib.request
import zipfile

import pandas as pd

AW_URL = ("https://github.com/microsoft/sql-server-samples/releases/download/"
          "adventureworks/AdventureWorksDW-data-warehouse-install-script.zip")
AW_TABLES = ["FactInternetSales", "DimCustomer", "DimGeography", "DimSalesTerritory", "DimProduct",
             "DimProductSubcategory", "DimProductCategory", "DimDate", "DimPromotion", "DimCurrency"]


def fetch_adventureworks(folder="adventureworks_dw"):
    """Download Microsoft's AdventureWorks sample warehouse once, and save each table as a CSV file with a header."""
    if os.path.isdir(folder):
        return folder
    archive = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(AW_URL).read()))
    script = archive.read("instawdbdw.sql")
    script = script.decode("utf-16" if script[:2] in (b"\xff\xfe", b"\xfe\xff") else "utf-8-sig")
    os.makedirs(folder)
    for table in AW_TABLES:
        # The column names are not in the data files: they are in the CREATE TABLE statements of the install script.
        body = re.search(r"CREATE TABLE \[dbo\]\.\[" + table + r"\]\((.*?)\n\)", script, re.S).group(1)
        columns = re.findall(r"^\s*\[(\w+)\]", body, re.M)
        text = archive.read(table + ".csv").decode("utf-8-sig")
        df = pd.read_csv(io.StringIO(text), sep="|", header=None, names=columns, dtype=str, keep_default_na=False)
        df = df.drop(columns=[c for c in ("LargePhoto", "SalesTerritoryImage") if c in df])   # pictures as hex text
        df.replace("", None).to_csv(f"{folder}/{table}.csv", index=False)
    return folder


def load(name, folder=None):
    """Read one table into pandas. Other exports of this warehouse are sometimes saved as Latin-1, so try both."""
    path = f"{folder or DATA}/{name}.csv"
    try:
        return pd.read_csv(path, encoding="utf-8-sig", low_memory=False)
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding="latin-1", low_memory=False)


DATA = fetch_adventureworks()
print("Tables:", sorted(f[:-4] for f in os.listdir(DATA)))

**What you should see:** a list of ten table names, from `DimCurrency` to `FactInternetSales`. The first run takes a few seconds because it downloads the data; after that the files are already in the Colab session.

## 1. SQLite from Python, from zero

### Intuition first
A CSV file is a sheet of paper: anyone can write anything on it, in any order, and nothing checks it. A **database** is more like an office with a clerk at the door. You give the clerk a request in a language called **SQL**, and the clerk stores the data, refuses requests that break the rules, and answers questions about what is stored.

In Python, talking to SQLite always follows the same four steps:

1. **connect** to a database (a file, or a temporary one in memory);
2. **execute** SQL statements: `CREATE TABLE` to make a table, `INSERT` to add rows, `SELECT` to ask questions;
3. **fetch** the answer, or let pandas read it into a DataFrame;
4. **commit** to save the changes, and **close** when you are done.

We first try these steps on a tiny table of our own, so that nothing else distracts us. The real warehouse comes in Section 3.

In [ ]:
import sqlite3
import time

toy = sqlite3.connect(":memory:")      # ":memory:" = a temporary database that lives only in RAM
toy.execute('''
    CREATE TABLE bikes (
        bike_id INTEGER PRIMARY KEY,   -- a number that identifies each bike; no two rows may share it
        name    TEXT,
        price   REAL                   -- REAL means a decimal number
    )''')
print("The table exists. Rows in it:", toy.execute("SELECT COUNT(*) FROM bikes").fetchone()[0])

**What you should see:** `The table exists. Rows in it: 0`.

Read the cell line by line. `sqlite3.connect` opens the database and gives us a **connection**, which we call `toy`. `toy.execute(...)` sends one SQL statement. `CREATE TABLE` names the table and lists its columns, each with a type. The text after `--` is an SQL comment. Finally we asked a question, `SELECT COUNT(*) FROM bikes` (how many rows?), and `.fetchone()[0]` took the first value of the first row of the answer.

Now add some rows. There are two ways to put values into an `INSERT`: written into the SQL text, or passed separately with `?` placeholders. Use the placeholders: they handle quotes and types for you, and they protect a real application from SQL injection.

In [ ]:
toy.execute("INSERT INTO bikes VALUES (1, 'Road-150', 3578.27)")                 # values written in the SQL
toy.execute("INSERT INTO bikes VALUES (?, ?, ?)", (2, "Mountain-100", 3399.99))   # values passed with ?
toy.executemany("INSERT INTO bikes VALUES (?, ?, ?)",                              # many rows at once
                [(3, "Touring-1000", 2384.07), (4, "Road-750", 539.99), (5, "Mountain-500", 539.99)])
toy.commit()                                                                       # make the changes permanent

answer = toy.execute("SELECT name, price FROM bikes WHERE price > 1000 ORDER BY price DESC")
answer.fetchall()

**What you should see:** a Python list of three pairs, from `('Road-150', 3578.27)` down to `('Touring-1000', 2384.07)`.

The question in plain English was: *give me the name and price of the bikes that cost more than 1000, most expensive first*. `SELECT` picks the columns, `WHERE` picks the rows, `ORDER BY ... DESC` sorts from high to low. `fetchall()` returns every row of the answer as a list of tuples.

Tuples are fine for a computer but not pleasant to read. pandas can run the same question and give back a DataFrame:

In [ ]:
import pandas as pd

pd.read_sql_query("SELECT name, price FROM bikes ORDER BY price", toy)

**What you should see:** a small table with all five bikes, cheapest first. From now on we use `pd.read_sql_query` whenever we want to *look* at an answer.

Now the most important idea of this notebook. Try to add a sixth bike with an id that is already taken:

In [ ]:
try:
    toy.execute("INSERT INTO bikes VALUES (?, ?, ?)", (1, "Another bike", 99.0))
except sqlite3.IntegrityError as error:
    print("The database refused the row:", error)

print("Rows still in the table:", toy.execute("SELECT COUNT(*) FROM bikes").fetchone()[0])

**What you should see:** `The database refused the row: UNIQUE constraint failed: bikes.bike_id`, and then `Rows still in the table: 5`.

`try` / `except` means: *try this; if this particular error happens, do not crash, run the `except` part instead*. The error came from the database, not from Python: because `bike_id` is the **primary key**, the database will never hold two rows with the same id. A CSV file would have accepted the duplicate without a word. This is why we move data into a database: the rules are written down once and then enforced on every row, forever.

That is all the SQLite we need. Close the toy database and turn to the real one.

In [ ]:
toy.close()

## 2. Choosing what goes in

### Intuition first
A warehouse is not a dump of every column that exists. Each column costs storage, makes every query that reads it a little slower, and, if it is personal data, carries a legal duty. So we choose: from the fact table, the keys and the measures (the numbers we will add up); from each dimension, the key and the descriptive columns that an analyst would group or filter by. As in Notebook 13, names, addresses and contact details stay out, and so do the Spanish and French translations.

The cell below is long but repetitive: each line loads one table with the `load` function from the first cell and keeps a list of columns.

In [ ]:
sales = load("FactInternetSales")
fact = sales[["SalesOrderNumber", "SalesOrderLineNumber", "ProductKey", "CustomerKey", "SalesTerritoryKey",
              "PromotionKey", "OrderDateKey", "ShipDateKey", "DueDateKey",
              "OrderQuantity", "UnitPrice", "TotalProductCost", "SalesAmount", "TaxAmt", "Freight"]]

dim_date = load("DimDate")[["DateKey", "FullDateAlternateKey", "EnglishDayNameOfWeek", "EnglishMonthName",
                            "MonthNumberOfYear", "CalendarQuarter", "CalendarYear"]]
dim_customer = load("DimCustomer")[["CustomerKey", "GeographyKey", "BirthDate", "MaritalStatus", "Gender",
                                    "YearlyIncome", "TotalChildren", "EnglishEducation", "EnglishOccupation",
                                    "HouseOwnerFlag", "NumberCarsOwned", "CommuteDistance"]]
dim_geography = load("DimGeography")[["GeographyKey", "City", "StateProvinceName", "EnglishCountryRegionName"]]
dim_territory = load("DimSalesTerritory")[["SalesTerritoryKey", "SalesTerritoryRegion", "SalesTerritoryCountry",
                                           "SalesTerritoryGroup"]]
dim_promotion = load("DimPromotion")[["PromotionKey", "EnglishPromotionName", "DiscountPct", "EnglishPromotionType"]]
dim_product = load("DimProduct")[["ProductKey", "ProductSubcategoryKey", "EnglishProductName", "Color",
                                  "StandardCost", "ListPrice", "ProductLine", "ModelName"]]
dim_subcategory = load("DimProductSubcategory")[["ProductSubcategoryKey", "EnglishProductSubcategoryName",
                                                 "ProductCategoryKey"]]
dim_category = load("DimProductCategory")[["ProductCategoryKey", "EnglishProductCategoryName"]]

for name, df in [("fact", fact), ("date", dim_date), ("customer", dim_customer), ("product", dim_product)]:
    print(f"{name:<10} {df.shape[0]:>6} rows x {df.shape[1]:>2} columns")

**What you should see:** the fact table with 60,398 rows and 15 columns, 3,652 dates (ten years of days), 18,484 customers and 606 products.

One column needs a repair before it goes into the database. In Notebook 13 we found that a third of the products have no subcategory. pandas stores a missing number as `NaN`, which only exists for decimal numbers, so the whole `ProductSubcategoryKey` column arrived as decimals: `1.0`, `2.0` and so on. A key should be a whole number. pandas has a special whole-number type, `Int64` with a capital I, that can also hold a missing value (shown as `<NA>`):

In [ ]:
dim_product["ProductSubcategoryKey"] = dim_product["ProductSubcategoryKey"].astype("Int64")

print("Missing subcategory keys:", dim_product["ProductSubcategoryKey"].isna().sum(), "of", len(dim_product))
dim_product[["EnglishProductName", "ProductSubcategoryKey"]].iloc[[0, 1, -3, -2, -1]]

**What you should see:** `Missing subcategory keys: 209 of 606`, and a table of five products. The first two (`Adjustable Race`, `Bearing Ball`) show `<NA>`: they are spare parts that the shop never sells online. The last three (`Road-750 Black` in three sizes) show the whole number `2`, no longer `2.0`.

`.iloc[[0, 1, -3, -2, -1]]` picks rows by position: the first two and the last three. We did not use `.head()` here on purpose: the first 209 products are exactly the ones without a subcategory, so `.head()` would show only `<NA>` and hide the repair.

## 3. Tables with rules

### Intuition first
In Notebook 13 we *checked* that every key had a partner. A database can do better: it can *refuse* any row that would break the rule, so the check never has to be repeated. You already saw the **primary key** do this for duplicates. Its partner is the **foreign key**: a column declared with `REFERENCES other_table(column)` may only hold a value that exists in that other table. A sale of product 99999 is refused if there is no product 99999.

We write the `CREATE TABLE` statements ourselves rather than letting pandas invent them, because pandas creates tables **without any keys**.

The cell below creates nine tables. Before you run it, read just two of them:

```sql
CREATE TABLE dim_product (
    product_key INTEGER PRIMARY KEY,                                        -- one row per product
    subcategory_key INTEGER REFERENCES dim_subcategory(subcategory_key),     -- must exist in dim_subcategory
    product TEXT, color TEXT, standard_cost REAL, list_price REAL, product_line TEXT, model TEXT);

CREATE TABLE fact_sales (
    order_number TEXT, line_number INTEGER,
    product_key INTEGER NOT NULL REFERENCES dim_product(product_key),       -- NOT NULL: every sale has a product
    ...
    PRIMARY KEY (order_number, line_number));                               -- the grain: one row per order line
```

The fact table's primary key is a **pair** of columns, the pair that Notebook 13 proved unique. The grain is now written into the database itself.

In [ ]:
if os.path.exists("adventureworks.db"):     # start from an empty file every time this cell runs
    os.remove("adventureworks.db")
dw = sqlite3.connect("adventureworks.db")      # dw = "data warehouse"; this time a real file
dw.execute("PRAGMA foreign_keys = ON")          # SQLite checks foreign keys only when asked to!

dw.executescript('''
CREATE TABLE dim_date (
    date_key INTEGER PRIMARY KEY, full_date TEXT, day_name TEXT, month_name TEXT,
    month INTEGER, quarter INTEGER, year INTEGER);
CREATE TABLE dim_territory (
    territory_key INTEGER PRIMARY KEY, region TEXT, country TEXT, territory_group TEXT);
CREATE TABLE dim_geography (
    geography_key INTEGER PRIMARY KEY, city TEXT, state TEXT, country TEXT);
CREATE TABLE dim_customer (
    customer_key INTEGER PRIMARY KEY,
    geography_key INTEGER REFERENCES dim_geography(geography_key),
    birth_date TEXT, marital_status TEXT, gender TEXT, yearly_income REAL, total_children INTEGER,
    education TEXT, occupation TEXT, house_owner INTEGER, cars_owned INTEGER, commute_distance TEXT);
CREATE TABLE dim_category (
    category_key INTEGER PRIMARY KEY, category TEXT);
CREATE TABLE dim_subcategory (
    subcategory_key INTEGER PRIMARY KEY, subcategory TEXT,
    category_key INTEGER REFERENCES dim_category(category_key));
CREATE TABLE dim_product (
    product_key INTEGER PRIMARY KEY,
    subcategory_key INTEGER REFERENCES dim_subcategory(subcategory_key),
    product TEXT, color TEXT, standard_cost REAL, list_price REAL, product_line TEXT, model TEXT);
CREATE TABLE dim_promotion (
    promotion_key INTEGER PRIMARY KEY, promotion TEXT, discount_pct REAL, promotion_type TEXT);
CREATE TABLE fact_sales (
    order_number TEXT, line_number INTEGER,
    product_key   INTEGER NOT NULL REFERENCES dim_product(product_key),
    customer_key  INTEGER NOT NULL REFERENCES dim_customer(customer_key),
    territory_key INTEGER NOT NULL REFERENCES dim_territory(territory_key),
    promotion_key INTEGER NOT NULL REFERENCES dim_promotion(promotion_key),
    order_date_key INTEGER NOT NULL REFERENCES dim_date(date_key),
    ship_date_key  INTEGER NOT NULL REFERENCES dim_date(date_key),
    due_date_key   INTEGER NOT NULL REFERENCES dim_date(date_key),
    quantity INTEGER, unit_price REAL, cost REAL, revenue REAL, tax REAL, freight REAL,
    PRIMARY KEY (order_number, line_number));
''')
print([r[0] for r in dw.execute("SELECT name FROM sqlite_master WHERE type = 'table'")])

**What you should see:** a list of nine table names, from `dim_date` to `fact_sales`.

Three details are worth a second look:

* `executescript` runs many statements at once; `execute` runs only one.
* `PRAGMA foreign_keys = ON` is a quirk of SQLite: for historical reasons it does **not** check foreign keys unless you switch the check on for each connection. Forget this line and every `REFERENCES` above is silently ignored.
* `sqlite_master` is a table that SQLite keeps about itself: it lists every table, index and view. Asking it is how we confirmed that the tables exist.

Look at `fact_sales` once more: three of its columns point to the **same** table, `dim_date`. The order date, the ship date and the due date are all looked up in one date table. One dimension playing several roles is called a **role-playing dimension**.

## 4. Loading in the right order

### Intuition first
A foreign key can only point to a row that already exists. You cannot record a sale of a product before the product is in the catalogue. So the tables must be loaded **from the outside in**: first the dimensions that point to nothing (dates, territories, categories), then those that point to them (subcategories, products, customers), and the fact table last.

`DataFrame.to_sql` copies a DataFrame into a database table. With `if_exists="append"` it adds rows to the table we created, and keeps our keys. The helper `put` also renames the columns to the short names of our tables, and prints how many rows went in.

In [ ]:
def put(table, df, columns):
    df.set_axis(columns, axis=1).to_sql(table, dw, if_exists="append", index=False)
    print(f"{table:<16} {len(df):>6} rows")

put("dim_date", dim_date, ["date_key", "full_date", "day_name", "month_name", "month", "quarter", "year"])
put("dim_territory", dim_territory, ["territory_key", "region", "country", "territory_group"])
put("dim_geography", dim_geography, ["geography_key", "city", "state", "country"])
put("dim_customer", dim_customer, ["customer_key", "geography_key", "birth_date", "marital_status", "gender",
                                   "yearly_income", "total_children", "education", "occupation", "house_owner",
                                   "cars_owned", "commute_distance"])
put("dim_category", dim_category, ["category_key", "category"])
put("dim_subcategory", dim_subcategory, ["subcategory_key", "subcategory", "category_key"])
put("dim_product", dim_product, ["product_key", "subcategory_key", "product", "color", "standard_cost",
                                 "list_price", "product_line", "model"])
put("dim_promotion", dim_promotion, ["promotion_key", "promotion", "discount_pct", "promotion_type"])
put("fact_sales", fact, ["order_number", "line_number", "product_key", "customer_key", "territory_key",
                         "promotion_key", "order_date_key", "ship_date_key", "due_date_key", "quantity",
                         "unit_price", "cost", "revenue", "tax", "freight"])
dw.commit()

**What you should see:** nine lines, from `dim_date 3652 rows` to `fact_sales 60398 rows`, with 4 categories, 37 subcategories and 606 products in between.

No error appeared, so the database accepted every row: every key found its partner. SQLite can also confirm it in one statement. An **empty list** means that no row breaks any foreign key:

In [ ]:
dw.execute("PRAGMA foreign_key_check").fetchall()

**What you should see:** `[]`, an empty list.

Now see the rules at work. Try to record a sale of product 99999, which does not exist:

In [ ]:
try:
    dw.execute('''INSERT INTO fact_sales VALUES ('SO99999', 1, 99999, 11000, 1, 1, 20130101, 20130105, 20130110,
                                                 1, 10.0, 5.0, 10.0, 0.8, 0.3)''')
except sqlite3.IntegrityError as error:
    print("Refused:", error)

**What you should see:** `Refused: FOREIGN KEY constraint failed`.

The bad row never enters the warehouse. That is the difference between *checking* quality (Notebook 13) and *enforcing* it.

> ⚠️ Many cloud warehouses (Redshift, BigQuery, Synapse) let you *declare* primary and foreign keys but do **not enforce** them: they use them only as hints for the query optimiser, because checking every key on billions of rows is expensive. There, the checks of Notebook 13 remain your job, usually as a step of the loading pipeline.

## 5. Star or snowflake?

### Intuition first
Draw the warehouse on paper: the fact table in the middle, the dimensions around it, a line for every foreign key. In a pure **star**, every dimension is a single table joined straight to the facts. Our product dimension is not: a product points to a subcategory, which points to a category. A dimension split into a chain of tables is called a **snowflake**, after the shape of the drawing. A snowflake stores each category name once, at the price of two extra joins in every query that groups by category.

The database can draw the picture for us in words: `PRAGMA foreign_key_list(table)` lists the foreign keys of a table.

In [ ]:
for (t,) in dw.execute("SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name"):
    links = [f"{fk[3]} -> {fk[2]}" for fk in dw.execute(f"PRAGMA foreign_key_list({t})")]
    print(f"{t:<16}", ", ".join(dict.fromkeys(links)) or "(no foreign keys)")

**What you should see:** five tables with `(no foreign keys)`, and four with arrows. `fact_sales` points to six dimensions, three times to `dim_date`. Two chains form the snowflake's arms: `dim_product -> dim_subcategory -> dim_category` and `dim_customer -> dim_geography`. Exercise 9 flattens the product chain into a star and measures the difference.

## 6. A view for the analysts

### Intuition first
Almost every business question about this warehouse needs the same joins: from a sale to its date, product, subcategory, category, customer and country. Writing seven joins for every question is tedious and error-prone. A **view** stores a query under a name, so that analysts can treat it as one flat table. Nothing is copied: the joins run again each time the view is read.

A `JOIN ... ON` pairs each row of one table with the row of another table whose key matches. `AS f`, `AS d` and so on are short nicknames for the tables, so that `f.revenue` means *the revenue column of fact_sales*.

In [ ]:
dw.executescript('''
DROP VIEW IF EXISTS v_sales;
CREATE VIEW v_sales AS
SELECT f.order_number, f.revenue, f.cost, f.quantity,
       d.full_date, d.year, d.quarter, d.month,
       p.product, s.subcategory, c.category,
       cu.gender, cu.yearly_income, cu.education, g.country
FROM fact_sales AS f
JOIN dim_date AS d        ON d.date_key = f.order_date_key
JOIN dim_product AS p     ON p.product_key = f.product_key
JOIN dim_subcategory AS s ON s.subcategory_key = p.subcategory_key
JOIN dim_category AS c    ON c.category_key = s.category_key
JOIN dim_customer AS cu   ON cu.customer_key = f.customer_key
JOIN dim_geography AS g   ON g.geography_key = cu.geography_key;
''')

def show(sql, **kw):
    """Run a query on the warehouse and return the answer as a DataFrame."""
    return pd.read_sql_query(sql, dw, **kw)

show("SELECT * FROM v_sales LIMIT 3")

**What you should see:** three sales from the end of December 2010: a `Road-150 Red, 62` bought in Canada and two `Mountain-100 Silver, 44` bought in France and the United States, each with its subcategory, category, the customer's gender, income and education, and the country. One flat row per sale, with no join in sight.

`LIMIT 3` asks for the first three rows only. Use it whenever you just want to see what a table or view looks like.

Now three classic moves of analysis. **Roll up**: from single sales up to one total per year. `GROUP BY year` makes one group per year, and `COUNT(*)` and `SUM(revenue)` summarise each group.

In [ ]:
show("SELECT year, COUNT(*) AS lines, ROUND(SUM(revenue)) AS revenue FROM v_sales GROUP BY year ORDER BY year")

**What you should see:** five years. 2010 has only 14 order lines and 2014 only 1,970, while 2013 has 52,801 lines and about 16.4 million in revenue. Remember from Notebook 13 that the first and the last year are not complete, so do not compare them with the others (Exercise 5 asks how you would check this).

**Drill down**: go one level deeper, from a year to its quarters, and split by category. `WHERE year = 2013` keeps one year only.

In [ ]:
show('''SELECT quarter, category, ROUND(SUM(revenue)) AS revenue
        FROM v_sales WHERE year = 2013
        GROUP BY quarter, category ORDER BY quarter, revenue DESC''')

**What you should see:** twelve rows, three categories in each of four quarters. Bikes lead every quarter by far (from about 2.5 million in the first quarter to about 5.0 million in the fourth), followed by accessories and then clothing. Revenue grows every quarter of the year.

**Slice**: fix one value of a dimension, here one country, and compute a measure that needs two columns. The gross margin is the share of revenue left after the cost of the products.

In [ ]:
show('''SELECT category, ROUND(SUM(revenue)) AS revenue,
               ROUND(100.0 * (SUM(revenue) - SUM(cost)) / SUM(revenue), 1) AS margin_pct
        FROM v_sales WHERE country = 'Germany'
        GROUP BY category ORDER BY revenue DESC''')

**What you should see:** in Germany, bikes bring about 2.8 million at a margin of about 41%, accessories about 62 thousand at about 63%, clothing about 24 thousand at 33%.

Read it as a manager would: bikes bring almost all of the money, but each accessory sold keeps a much larger share of its price. A sentence like that is the purpose of the whole warehouse.

## 7. How fast, and why?

### Intuition first
To find one word in a book without an index, you read every page. With the index at the back, you jump straight to the right page. A database **index** is the same idea: a sorted list of the values of a column with the position of each row. The database keeps one automatically for every primary key.

`EXPLAIN QUERY PLAN` asks the database *how* it intends to answer a question, without answering it. `SCAN` means "I will read every row". `SEARCH ... USING INDEX` means "I will jump". The helper `timed` runs a query many times and returns the average time in milliseconds.

Our question: the total spent by one customer.

In [ ]:
one_customer = "SELECT SUM(revenue) FROM fact_sales WHERE customer_key = 11000"

def plan(sql):
    return [row[-1] for row in dw.execute("EXPLAIN QUERY PLAN " + sql)]

def timed(sql, repeats=50):
    started = time.perf_counter()
    for i in range(repeats):
        dw.execute(sql).fetchall()
    return (time.perf_counter() - started) / repeats * 1000

print("before:", plan(one_customer), f"{timed(one_customer):.2f} ms")
dw.execute("CREATE INDEX ix_sales_customer ON fact_sales(customer_key)")
print("after: ", plan(one_customer), f"{timed(one_customer):.2f} ms")

**What you should see:** before the index, a plan with `SCAN fact_sales` and a few milliseconds. After it, `SEARCH fact_sales USING INDEX ix_sales_customer (customer_key=?)` and a few hundredths of a millisecond: around a hundred times faster. Your exact times depend on the machine, but the gap should be large.

So should we index everything? A warehouse is mostly asked the *other* kind of question, the one that has to read every row:

In [ ]:
by_category = "SELECT category, ROUND(SUM(revenue), 2) FROM v_sales GROUP BY category"
print(f"{timed(by_category, 5):.0f} ms per run of the full analytical query")

**What you should see:** somewhere around 50 to 150 milliseconds; the exact number changes from run to run and from machine to machine. That is long compared with the single customer, because the query joins and reads all 60,398 sales.

No index helps a question that needs every row: there is nothing to jump to. What does help is to compute the answer once and keep it, in a **summary table** (also called an *aggregate table*). Reports that are opened every morning are typically served this way.

In [ ]:
dw.executescript('''
DROP TABLE IF EXISTS sum_month_category;
CREATE TABLE sum_month_category AS
SELECT year, month, category, SUM(revenue) AS revenue, SUM(cost) AS cost, COUNT(*) AS lines
FROM v_sales GROUP BY year, month, category;
''')
from_summary = "SELECT category, ROUND(SUM(revenue), 2) FROM sum_month_category GROUP BY category"
print("rows in the summary:", dw.execute("SELECT COUNT(*) FROM sum_month_category").fetchone()[0])
print(f"from the facts:  {timed(by_category, 5):.1f} ms")
print(f"from the summary: {timed(from_summary):.2f} ms")
print("same answer:", sorted(dw.execute(by_category).fetchall()) == sorted(dw.execute(from_summary).fetchall()))

**What you should see:** 65 rows in the summary, tens of milliseconds from the facts against a fraction of a millisecond from the summary, and `same answer: True`. That is several hundred times faster for the same result.

The price: the summary must be rebuilt whenever new sales arrive (Exercise 10), and it can only answer the questions it was built for. It knows months and categories, but nothing about countries. Cloud warehouses offer the same trade under the name **materialised view**.

## ⚠️ Common pitfalls

* **Forgetting `PRAGMA foreign_keys = ON`.** SQLite then accepts every broken key without complaint. Switch it on for every new connection.
* **`to_sql(..., if_exists="replace")` on a table you created yourself.** pandas drops your table and makes a new one *without* your keys. Use `"append"` to load into your own tables.
* **Loading the fact table first.** Every row is refused, because its keys point to dimensions that are still empty.
* **Counting rows when you mean orders.** The grain is the order *line*: `COUNT(*)` counts lines, `COUNT(DISTINCT order_number)` counts orders.
* **Trusting a summary table after new data arrived.** It still shows the old totals until it is rebuilt.
* **Forgetting `commit()`.** Changes that were never committed are lost when the connection closes.

## 8. Where we stand

| The need | The tool |
|---|---|
| Talk to a database from Python | `sqlite3.connect`, `execute`, `fetchall`, `pd.read_sql_query`, `commit` |
| The grain must stay true | a primary key on the fact table |
| Every key must find its partner | foreign keys, switched on, and loading from the outside in |
| One date table for several dates of an event | a role-playing dimension |
| Category names stored once | a snowflaked dimension, at the cost of extra joins |
| Analysts should not write joins | a view |
| Fast lookups of a few rows | an index |
| Fast answers to the same big question | a summary table (a materialised view) |

The warehouse is relational to the core: fixed tables, fixed columns, joins. The next notebook asks what the same data looks like when it is stored the way a document database would store it, and what that gains and loses.

---
## ✏️ Exercises

## Level 1: warm-up

Short questions on one table or the view. Each needs one `SELECT`.

### Exercise 1 (Counting)
How many products, customers and promotions are in the warehouse? Use `COUNT(*)` on each table, once with `dw.execute(...).fetchone()` and once with `show(...)`.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
for table in ["dim_product", "dim_customer", "dim_promotion"]:
    print(table, dw.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0])

show("SELECT COUNT(*) AS products FROM dim_product")
```

*606 products, 18,484 customers and 16 promotions. fetchone gives a bare number for code to use; show gives a small table for people to read.*

</details>

### Exercise 2 (The categories)
List every category with its key. Then list the subcategories of category 1 (`WHERE category_key = 1`), in alphabetical order.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
display(show("SELECT * FROM dim_category"))
show("SELECT subcategory FROM dim_subcategory WHERE category_key = 1 ORDER BY subcategory")
```

*Four categories: Bikes, Components, Clothing and Accessories. Category 1, Bikes, has three subcategories: Mountain Bikes, Road Bikes and Touring Bikes.*

</details>

### Exercise 3 (One number from the view)
Using `v_sales`, what was the total revenue in 2012, and how many order lines produced it?

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
show("SELECT COUNT(*) AS lines, ROUND(SUM(revenue)) AS revenue FROM v_sales WHERE year = 2012")
```

*3,397 lines and about 5.84 million. Compare it with 2011 (2,216 lines, about 7.08 million): less revenue from more lines, a first hint of the cheaper products that arrive in 2013.*

</details>

## Level 2: think it through

These questions are about **reasoning**: before you write code, decide what you would need to see to answer the question. Then write the query that shows it.

### Exercise 4 (Does the view lose sales?)
`v_sales` uses plain `JOIN`s, which keep only the rows that find a partner. If a sold product had no subcategory, its sales would silently disappear from every report built on the view, and totals would be too low. **What two numbers would you compare to find out whether any sale is lost?** Compute them. Then write a query that would *list* the lost sales if there were any.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
print("rows in fact_sales:", dw.execute("SELECT COUNT(*) FROM fact_sales").fetchone()[0])
print("rows in v_sales:   ", dw.execute("SELECT COUNT(*) FROM v_sales").fetchone()[0])

# A LEFT JOIN keeps every sale; where the partner is missing, its columns are NULL.
show('''SELECT f.order_number, f.product_key
        FROM fact_sales AS f
        LEFT JOIN dim_product AS p     ON p.product_key = f.product_key
        LEFT JOIN dim_subcategory AS s ON s.subcategory_key = p.subcategory_key
        WHERE s.subcategory_key IS NULL''')
```

*Both counts are 60,398 and the list is empty, so no sale is lost: the 209 products without a subcategory were never sold online. Comparing the row count before and after a join is the quickest test there is, and it is worth running every time you build a view.*

</details>

### Exercise 5 (Did sales collapse in 2014?)
The roll-up shows 1,970 lines in 2014 against 52,801 in 2013. A manager asks whether sales collapsed. **What would you check before you answer?** Write one query that shows, for each year, the first and the last order date and the number of different months with sales.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
show('''SELECT year, MIN(full_date) AS first_day, MAX(full_date) AS last_day,
               COUNT(DISTINCT month) AS months
        FROM v_sales GROUP BY year ORDER BY year''')
```

*The data for 2014 stops at the end of January: one month, against twelve in 2013. Sales did not collapse; the data ends. Always check how much of a period the data covers before comparing periods.*

</details>

### Exercise 6 (Predict, then measure)
Before running anything, **write down a prediction**: will an index on `fact_sales(product_key)` make the query `by_category` (revenue by category, from Section 7) faster? Give a reason. Then create the index and measure.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
def best(sql):
    return min(timed(sql, 5) for _ in range(3))   # times are noisy: take the best of three measurements

print("plan before:", plan(by_category)[0], f"  time: {best(by_category):.0f} ms")
dw.execute("CREATE INDEX ix_sales_product ON fact_sales(product_key)")
print("plan after: ", plan(by_category)[0], f"  time: {best(by_category):.0f} ms")
```

*The plan does not change: it still starts with SCAN fact_sales, because the question needs every sale and there is nothing to jump to. Any difference in time is noise; run the cell again and it moves either way. When a timing surprises you, read the plan before you believe the clock.*

</details>

### Exercise 7 (Which rule refuses this row?)
Each of these rows would be refused by the warehouse. For each one, **say which rule refuses it before you run it**, then check with `try` / `except`: (a) a second copy of the order line `('SO43697', 1, ...)` with the same values as the original; (b) a customer with `geography_key = 99999`; (c) a sale with `product_key = NULL`.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
attempts = {
    "(a)": "INSERT INTO fact_sales SELECT * FROM fact_sales WHERE order_number = 'SO43697' AND line_number = 1",
    "(b)": "INSERT INTO dim_customer (customer_key, geography_key) VALUES (99999, 99999)",
    "(c)": '''INSERT INTO fact_sales VALUES ('SO99998', 1, NULL, 11000, 1, 1, 20130101, 20130105, 20130110,
                                            1, 10.0, 5.0, 10.0, 0.8, 0.3)''',
}
for label, sql in attempts.items():
    try:
        dw.execute(sql)
        print(label, "accepted (unexpected!)")
    except sqlite3.IntegrityError as error:
        print(label, "refused:", error)
```

*(a) breaks the primary key of the grain (UNIQUE constraint failed), (b) breaks the foreign key to dim_geography, and (c) breaks NOT NULL. Three different rules, three different messages: reading the message tells you which rule a bad row broke.*

</details>

## Level 3: technical

Longer tasks that combine several ideas of the notebook.

### Exercise 8 (Three business questions)
Using `v_sales`, answer: (a) the five best-selling **products** by revenue in 2013; (b) the average revenue **per order** in each country; (c) the share of revenue from each **education** level.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
display(show('''SELECT product, ROUND(SUM(revenue)) AS revenue FROM v_sales WHERE year = 2013
                GROUP BY product ORDER BY revenue DESC LIMIT 5'''))
display(show('''SELECT country, ROUND(SUM(revenue) / COUNT(DISTINCT order_number), 1) AS revenue_per_order
                FROM v_sales GROUP BY country ORDER BY revenue_per_order DESC'''))
show('''SELECT education, ROUND(100.0 * SUM(revenue) / (SELECT SUM(revenue) FROM v_sales), 1) AS share_pct
        FROM v_sales GROUP BY education ORDER BY share_pct DESC''')
```

*In (b) the denominator is the number of distinct orders, not of rows: the grain of the fact table is the order line, so counting rows would give revenue per line.*

</details>

### Exercise 9 (From snowflake to star)
Create `dim_product_flat` with one row per product and the columns `product_key`, `product`, `subcategory` and `category` (a pure star dimension). Then time *revenue by category* through the snowflake and through the flat dimension, and check that both give the same answer. Look at the query plan of the star version too: is anything missing from the new table?

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
dw.executescript('''
DROP TABLE IF EXISTS dim_product_flat;
CREATE TABLE dim_product_flat AS
SELECT p.product_key, p.product, s.subcategory, c.category
FROM dim_product AS p
LEFT JOIN dim_subcategory AS s ON s.subcategory_key = p.subcategory_key
LEFT JOIN dim_category AS c    ON c.category_key = s.category_key;
''')
snowflake = '''SELECT c.category, ROUND(SUM(f.revenue), 2) FROM fact_sales AS f
               JOIN dim_product AS p ON p.product_key = f.product_key
               JOIN dim_subcategory AS s ON s.subcategory_key = p.subcategory_key
               JOIN dim_category AS c ON c.category_key = s.category_key GROUP BY c.category'''
star = '''SELECT p.category, ROUND(SUM(f.revenue), 2) FROM fact_sales AS f
          JOIN dim_product_flat AS p ON p.product_key = f.product_key GROUP BY p.category'''
best = lambda sql: min(timed(sql, 5) for _ in range(3))
print("star plan:", " | ".join(plan(star)))
print(f"snowflake: {best(snowflake):.0f} ms   star without a key: {best(star):.0f} ms")

# CREATE TABLE ... AS copies the data but not the keys: give the new table its key back.
dw.execute("CREATE UNIQUE INDEX ix_flat_product ON dim_product_flat(product_key)")
print("star plan:", " | ".join(plan(star + " ")))
print(f"snowflake: {best(snowflake):.0f} ms   star with a key:    {best(star + ' '):.0f} ms")
print("same answer:", sorted(dw.execute(snowflake).fetchall()) == sorted(dw.execute(star).fetchall()))
```

*Without a key the star is the slower of the two: the plan shows SQLite working around the missing key, either by building a temporary AUTOMATIC index on every run or by turning the join around. CREATE TABLE ... AS copies the data but never the keys. With the key restored, most of the gap closes (what remains moves from run to run), because the snowflake's two extra joins are against tables of only 37 and 4 rows. Here the star's real advantage is simplicity: one join instead of three for every analyst.*

</details>

### Exercise 10 (A summary that stays correct)
New sales arrive for March 2014. Insert one order line into `fact_sales` (any existing customer and product, order date 20140310, ship 20140317, due 20140322). Show that `sum_month_category` is now **stale**, then rebuild only the affected month: delete its rows from the summary and insert them again from `v_sales`.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
dw.execute('''INSERT INTO fact_sales VALUES ('SO99990', 1, 310, 11000, 6, 1, 20140310, 20140317, 20140322,
                                             1, 3578.27, 2171.29, 3578.27, 286.26, 89.46)''')
fresh = "SELECT ROUND(SUM(revenue), 2) FROM v_sales WHERE year = 2014 AND month = 3"
stale = "SELECT ROUND(SUM(revenue), 2) FROM sum_month_category WHERE year = 2014 AND month = 3"
print("facts:", dw.execute(fresh).fetchone()[0], "  summary:", dw.execute(stale).fetchone()[0])

dw.executescript('''
DELETE FROM sum_month_category WHERE year = 2014 AND month = 3;
INSERT INTO sum_month_category
SELECT year, month, category, SUM(revenue), SUM(cost), COUNT(*)
FROM v_sales WHERE year = 2014 AND month = 3 GROUP BY year, month, category;
''')
print("facts:", dw.execute(fresh).fetchone()[0], "  summary:", dw.execute(stale).fetchone()[0])
```

*Before the rebuild the summary shows None for March 2014, a month it has never seen, while the facts already hold the new sale; after it both agree. Rebuilding only the month that changed is the incremental, idempotent refresh of Notebooks 10 and 12: run it twice and the summary is still right.*

</details>

## 🔑 Key takeaways

- Working with SQLite from Python is always the same: **connect**, **execute** SQL, **fetch** or `pd.read_sql_query`, **commit**, **close**.
- Pass values with `?` placeholders, never by gluing text into SQL.
- Write the `CREATE TABLE` statements yourself: **primary keys** fix the grain and **foreign keys** keep every link valid. In SQLite, switch them on with `PRAGMA foreign_keys = ON`.
- Load **from the outside in**: dimensions first, the fact table last.
- Checking quality finds bad rows; **enforcing** it keeps them out. Many cloud warehouses declare keys without enforcing them.
- Compare row counts before and after a join: it is the fastest way to find lost data.
- A **view** hides the joins; an **index** speeds up questions about a few rows; a **summary table** speeds up big questions that are asked again and again, and must be refreshed.

**Next:** *Notebook 15: From Warehouse to Documents*, where the same sales are reshaped into the documents of Notebook 06, and we measure what the new shape makes easy and what it makes hard.

In [ ]:
dw.close()

---
*© Ioannis Tsioulis. *From One Table to a Cluster: Working with Data at Scale*. Prepared for educational use.*